# Engine v4 · Stockfish — the measurements → `results.csv`

Stockfish 16 with its network on and off

1. **The speed table** — three positions (Engine v3's), depth 16, NNUE on and off: the move, the score, the nodes searched, nodes per second and seconds. The network is slower per node and needs fewer nodes; this table says how much.
2. **The match** — NNUE against classical at the same fixed depth, ten openings each played twice with colors swapped (20 games): what the network is worth when both sides see equally far.
3. **One change** — the same match with one thing changed: give the classical side more depth (`CLASSICAL_DEPTH = 12`: how many plies is the network worth?), or give both sides a clock instead of a depth (`TIME = 0.05`: does classical's speed buy back what its evaluation lacks?).



In [1]:
%pip install chess -q
DEPTH           = 10      # depth for both sides in the match
PAIRS           = 10      # openings, each played twice -> 20 games
CLASSICAL_DEPTH = 12      # the change: classical searches this deep instead (None = no change)
NNUE_DEPTH      = None    # or: NNUE searches this deep instead
TIME            = None    # or: both sides get this many seconds a move, e.g. 0.05 (used instead of the depths)
OUT             = "engines/v4-stockfish/results.csv"

import os
if not os.path.isdir("scripts") and os.path.isdir(os.path.join("..", "scripts")):
    os.chdir("..")                      # opened from notebooks/ -- work from the portfolio root instead
print("working in", os.path.basename(os.getcwd()))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
working in content


## Setup — Stockfish with NNUE on or off, the positions, the openings, the match

In [3]:
import csv
import math
import os
import shutil
import sys
import time

import chess
import chess.engine

POSITIONS = {                                                     # the three positions of Engine v3's node table
    "start":   chess.STARTING_FEN,
    "italian": "r1bqk2r/pppp1ppp/2n5/8/1bBPn3/2N2N2/PP3PPP/R1BQK2R w KQkq - 0 8",
    "open":    "r2qkb1r/ppp2ppp/2n2n2/3pp3/4P1b1/3P1N2/PPP1BPPP/RNBQK2R w KQkq - 0 1",
}
OPENINGS = ["e4 e5", "d4 d5", "e4 c5", "d4 Nf6", "c4 e5", "e4 e6", "Nf3 d5", "e4 c6", "d4 f5", "g3 d5"]


def open_stockfish(path, nnue):
    eng = chess.engine.SimpleEngine.popen_uci(path)
    if "Use NNUE" not in eng.options:
        eng.quit()
        sys.exit("This Stockfish has no 'Use NNUE' option: use Stockfish 12-16 and set STOCKFISH_PATH")
    eng.configure({"Threads": 1, "Hash": 64, "Use NNUE": nnue})
    return eng


def speed_table(path, depth=16):
    rows = []
    for nnue in (True, False):
        eng = open_stockfish(path, nnue)
        for name, fen in POSITIONS.items():
            board = chess.Board(fen)
            eng.configure({"Clear Hash": None})
            t0 = time.perf_counter()
            info = eng.analyse(board, chess.engine.Limit(depth=depth))
            secs = time.perf_counter() - t0
            rows.append({"section": "speed", "position_or_match": f"{name} (depth {depth})",
                         "variant": "NNUE" if nnue else "classical", "move": board.san(info["pv"][0]),
                         "score": info["score"].white().score(mate_score=100000), "nodes": info["nodes"],
                         "nps": info["nodes" ] / secs, "seconds": round(secs, 2)})
        eng.quit()
    return rows


def match(path, nnue_limit, classical_limit, pairs=10, max_plies=240):
    """NNUE vs classical, each opening twice with colours swapped. Returns points for NNUE and W/D/L."""
    a, c = open_stockfish(path, True), open_stockfish(path, False)
    points, w, d, l = 0.0, 0, 0, 0
    try:
        for g in range(2 * pairs):
            nnue_white = g % 2 == 0
            board = chess.Board()
            for san in OPENINGS[(g // 2) % len(OPENINGS)].split():
                board.push_san(san)
            while not board.is_game_over(claim_draw=True) and len(board.move_stack) < max_plies:
                nnue_turn = (board.turn == chess.WHITE) == nnue_white
                eng, lim = (a, nnue_limit) if nnue_turn else (c, classical_limit)
                board.push(eng.play(board, lim).move)
            result = board.result(claim_draw=True) if board.is_game_over(claim_draw=True) else "1/2-1/2"
            s = {"1-0": 1.0, "0-1": 0.0, "1/2-1/2": 0.5}[result]
            s = s if nnue_white else 1 - s
            points += s
            w, d, l = w + (s == 1.0), d + (s == 0.5), l + (s == 0.0)
            print(f"    game {g + 1:>2}  {OPENINGS[(g // 2) % len(OPENINGS)]:<7} NNUE {'white' if nnue_white else 'black'}"
                  f"  {result:<7} {len(board.move_stack):>3} plies   NNUE has {points:g}/{g + 1}")
    finally:
        a.quit()
        c.quit()
    return points, w, d, l


def wilson(p, n, z=1.96):
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, centre - half), min(1.0, centre + half)


def limit_text(lim):
    return f"{lim.time:g} s a move" if lim.time else f"depth {lim.depth}"


STOCKFISH = os.environ.get("STOCKFISH_PATH") or shutil.which("stockfish") or "/usr/games/stockfish"

# Auto-install Stockfish in Colab if not found
if not os.path.isfile(STOCKFISH):
    print("Stockfish not found. Installing via apt-get...")
    os.system("sudo apt-get update && sudo apt-get install -y stockfish")
    STOCKFISH = shutil.which("stockfish") or "/usr/games/stockfish"

assert os.path.isfile(STOCKFISH), "Stockfish not found -- set STOCKFISH to its path and re-run"
print("Stockfish:", STOCKFISH)

Stockfish not found. Installing via apt-get...
Stockfish: /usr/games/stockfish


## Measurement 1 — the speed table

In [4]:
t_start = time.time()
print("1 -- THE SPEED TABLE: same position, same depth, NNUE on and off\n")
rows = speed_table(STOCKFISH)
print(f"  {'position':<20} {'eval':<10} {'move':<6} {'score':>6} {'nodes':>10} {'nodes/s':>10} {'secs':>6}")
for r in rows:
    print(f"  {r['position_or_match']:<20} {r['variant']:<10} {r['move']:<6} {r['score']:>+6} {r['nodes']:>10,}"
          f" {r['nps']:>10,.0f} {r['seconds']:>6.2f}")
    r["nps"] = round(r["nps"])


1 -- THE SPEED TABLE: same position, same depth, NNUE on and off

  position             eval       move    score      nodes    nodes/s   secs
  start (depth 16)     NNUE       e4        +24    307,700    262,663   1.17
  italian (depth 16)   NNUE       O-O       -54    213,813    326,925   0.65
  open (depth 16)      NNUE       Nbd2      -30    179,476    265,299   0.68
  start (depth 16)     classical  e4        +22    494,045    725,284   0.68
  italian (depth 16)   classical  O-O        +6    218,595    607,103   0.36
  open (depth 16)      classical  exd5       -4    301,584    564,706   0.53


## Measurements 2 and 3 — the match, and one change

Each game prints as it finishes. Twenty games give a wide 95% interval: quote it, and claim no more than it supports.

In [5]:
matches = [("NNUE vs classical, same depth", chess.engine.Limit(depth=DEPTH), chess.engine.Limit(depth=DEPTH))]
if TIME:
    matches.append((f"NNUE vs classical, {TIME:g} s a move", chess.engine.Limit(time=TIME),
                    chess.engine.Limit(time=TIME)))
elif CLASSICAL_DEPTH or NNUE_DEPTH:
    nd, cd = NNUE_DEPTH or DEPTH, CLASSICAL_DEPTH or DEPTH
    matches.append((f"NNUE depth {nd} vs classical depth {cd}", chess.engine.Limit(depth=nd), chess.engine.Limit(depth=cd)))

for k, (label, nl, cl) in enumerate(matches, start=2):
    print(f"\n{k} -- {'THE MATCH' if k == 2 else 'ONE CHANGE'}: {label} (NNUE {limit_text(nl)}, classical {limit_text(cl)})\n")
    pts, w, d, l = match(STOCKFISH, nl, cl, PAIRS)
    n = w + d + l
    lo, hi = wilson(pts / n, n)
    print(f"\n  NNUE scored {pts:g}/{n} ({w} wins, {d} draws, {l} losses); 95% interval for its score "
          f"{lo:.2f} to {hi:.2f}")
    rows.append({"section": "match", "position_or_match": label,
                 "variant": f"NNUE {limit_text(nl)} / classical {limit_text(cl)}", "games": n,
                 "points_for_nnue": pts, "wins": w, "draws": d, "losses": l})



2 -- THE MATCH: NNUE vs classical, same depth (NNUE depth 10, classical depth 10)

    game  1  e4 e5   NNUE white  1-0     137 plies   NNUE has 1/1
    game  2  e4 e5   NNUE black  0-1     166 plies   NNUE has 2/2
    game  3  d4 d5   NNUE white  1/2-1/2 136 plies   NNUE has 2.5/3
    game  4  d4 d5   NNUE black  1/2-1/2  30 plies   NNUE has 3/4
    game  5  e4 c5   NNUE white  1-0     157 plies   NNUE has 4/5
    game  6  e4 c5   NNUE black  0-1     184 plies   NNUE has 5/6
    game  7  d4 Nf6  NNUE white  1/2-1/2 126 plies   NNUE has 5.5/7
    game  8  d4 Nf6  NNUE black  1-0     149 plies   NNUE has 5.5/8
    game  9  c4 e5   NNUE white  1-0     139 plies   NNUE has 6.5/9
    game 10  c4 e5   NNUE black  0-1     160 plies   NNUE has 7.5/10
    game 11  e4 e6   NNUE white  0-1      74 plies   NNUE has 7.5/11
    game 12  e4 e6   NNUE black  0-1     168 plies   NNUE has 8.5/12
    game 13  Nf3 d5  NNUE white  1-0     163 plies   NNUE has 9.5/13
    game 14  Nf3 d5  NNUE black  1/2-1

## `results.csv`

In [6]:
os.makedirs(os.path.dirname(OUT), exist_ok=True)
fields = ["section", "position_or_match", "variant", "move", "score", "nodes", "nps", "seconds",
          "games", "points_for_nnue", "wins", "draws", "losses"]
with open(OUT, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
print(f"\nsaved {len(rows)} rows -> {OUT}   [{time.time() - t_start:.0f}s]")
print(open(OUT).read())
print(f"[{time.time() - t_start:.0f}s]")



saved 8 rows -> engines/v4-stockfish/results.csv   [186s]
section,position_or_match,variant,move,score,nodes,nps,seconds,games,points_for_nnue,wins,draws,losses
speed,start (depth 16),NNUE,e4,24,307700,262663,1.17,,,,,
speed,italian (depth 16),NNUE,O-O,-54,213813,326925,0.65,,,,,
speed,open (depth 16),NNUE,Nbd2,-30,179476,265299,0.68,,,,,
speed,start (depth 16),classical,e4,22,494045,725284,0.68,,,,,
speed,italian (depth 16),classical,O-O,6,218595,607103,0.36,,,,,
speed,open (depth 16),classical,exd5,-4,301584,564706,0.53,,,,,
match,"NNUE vs classical, same depth",NNUE depth 10 / classical depth 10,,,,,,20,14.5,12,5,3
match,NNUE depth 10 vs classical depth 12,NNUE depth 10 / classical depth 12,,,,,,20,13.5,8,11,1

[186s]
